# 1.1C - Persiapan Data: Lapisan Gold (Text Chunking & Dataset Vektor)

Notebook ini merupakan tahap ketiga dan final dalam pipeline persiapan data **Retrieval-Augmented Generation (RAG)** menggunakan arsitektur **Medallion Lakehouse (Bronze -> Silver -> Gold)**.

### Ringkasan Proses:
1. **Instalasi Pustaka Pemecah Teks**: Menginstal pustaka `langchain_text_splitters` untuk strategi chunking cerdas dan merestart kernel Python.
2. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, volume, dan variabel konfigurasi dari `../Includes/Lab_Setup`.
3. **Membaca Tabel Lapisan Silver**: Memuat teks dokumen terstruktur (`silver_docs_parsed`) dari tahap 1.1B.
4. **Konfigurasi Text Splitter**: Menyiapkan `RecursiveCharacterTextSplitter` dengan pemisahan berbasis batas halaman (`== page ==`), paragraf, dan ukuran batas karakter.
5. **Uji Coba Chunking Dokumen Sampel**: Melakukan validasi hasil chunking pada teks sampel dokumen spesifikasi produk.
6. **Pemrosesan Terdistribusi dengan PySpark Pandas UDF**: Menerapkan fungsi chunking secara paralel ke seluruh dokumen menggunakan `@pandas_udf` dan meratakannya menjadi baris per chunk dengan fungsi `explode`.
7. **Penambahan Primary Key & Penyimpanan ke Lapisan Gold**: Menambahkan kolom `id` unik dan menyimpan hasilnya ke tabel Delta terkelola `gold_docs_chunked` yang siap diindeks ke dalam **Databricks Vector Search** (notebook `1.2`).


### Langkah 1: Instalasi Library Pendukung
Menginstal paket `langchain_text_splitters` untuk memotong teks secara terstruktur dan merestart sesi Python agar perubahan library langsung aktif.


In [0]:
# Menginstal pustaka langchain_text_splitters untuk memecah teks dokumen secara cerdas
%pip install --quiet --upgrade langchain_text_splitters
# Merestart kernel Python agar pustaka yang baru diinstal dapat langsung digunakan
%restart_python

### Langkah 2: Setup Lingkungan Lab
Menjalankan notebook setup untuk menginisialisasi variabel konfigurasi lab (seperti objek `course`, katalog, skema, dan volume di Unity Catalog).


In [0]:
%run ../Includes/Lab_Setup

### Langkah 3: Membaca Tabel Lapisan Silver
Membaca tabel Delta `silver_docs_parsed` yang telah dibersihkan dan diformat ke Markdown pada tahap 1.1B.


In [0]:
# Membaca tabel Delta lapisan Silver yang berisi teks dokumen terstruktur (plain_text)
silver_df = spark.read.table("silver_docs_parsed")

# Menampilkan isi tabel untuk memverifikasi data sebelum proses chunking
display(silver_df)

### Langkah 4: Konfigurasi Recursive Character Text Splitter
Mendefinisikan strategi chunking menggunakan `RecursiveCharacterTextSplitter`:
- `CHUNK_SIZE = 1500`: batas maksimal panjang karakter pada tiap potongan teks.
- `CHUNK_OVERLAP = 150`: overlap 10% antar potongan teks untuk menjaga kesinambungan konteks semantik.
- `separators`: urutan prioritas pemisah, dimulai dari pemisah halaman (`== page ==`), pemisah paragraf ganda, baris baru, spasi, hingga karakter.


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ukuran target karakter untuk setiap potongan teks (chunk)
CHUNK_SIZE = 1500
# Jumlah karakter tumpang tindih antar chunk bertetangga untuk menjaga kontinuitas konteks semantik
CHUNK_OVERLAP = 150

# Inisialisasi RecursiveCharacterTextSplitter dengan hierarki pemisah teks:
# Prioritas utama adalah pemisah halaman '== page ==', diikuti paragraf ganda, baris baru, spasi, dan karakter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

def chunk_text(text):
    """Fungsi pembantu untuk memecah string teks menjadi list potongan teks (chunks) non-kosong."""
    chunks = []
    if isinstance(text, str) and text.strip():
        for c in splitter.split_text(text):
            if c and c.strip():
                chunks.append(c)
    return chunks

### Langkah 5: Pengujian Chunking pada Dokumen Sampel
Melakukan verifikasi (*dry-run*) fungsi `chunk_text` pada contoh teks dokumen produk "dPhone D1" untuk mengamati bagaimana dokumen dipotong berdasarkan batas halaman dan ukuran chunk.


In [0]:
# Contoh teks dokumen spesifikasi produk dengan beberapa halaman bertanda '== page =='
doc = """DerarTel MOBILE TECHNOLOGY ## Product Specifications: dPhone D1 Status: Internal/Confidential ### 1. Executive Summary The dPhone D1 is engineered to redefine the premium smartphone market by bridging the gap between high-performance computing and minimalist design. The D1 prioritizes Open Ecosystem integration, Adaptive Hardware, and Quantum-Safe Encryption. > dPhone D1 > Fluid OS > March 28, 2028 > dPhone D1 | Quantum-Safe Encryption > Invisible under-display tech > dPhone D1 | Matte 95% recycled titanium > Apex Display > Apex Secondary > Sealed dual-purpose microSD slot and Hydro-Guard Seal **Figure 1.** dPhone D1 product image showcasing the full design and display features == page == ### 2. Key Value Propositions The “Fluid” OS: A proprietary operating system built on a microkernel architecture that allows for 100% app compatibility with both Android and iOS environments. Sustainability: Built with a 93% recycled titanium chassis and a user‑replaceable battery. | Feature | Specification | |---------|---------------| | Processor | Helios G1M Neural Chip (5nm architecture) | | Display | 6.5" Super‑AMOLED “Infinity” Display | | Storage | 512GB | | Refresh Rate | Dynamic 1Hz to 144Hz | | Battery | 5,500 mAh with 60W Rapid Charging | | Security | Biometric “Vein‑Scan” (Under‑display) | | Connectivity | 6G Ready & Satellite SOS | ### 3. Environmental Resistance To ensure maximum durability, the dPhone D1 features industry‑leading waterproof protection: IP69 Rating: The highest rating for ingress protection, certifying the dPhone D1 is completely dust‑tight and capable of withstanding high‑pressure, high‑temperature water jets. Hydro‑Guard Internal Seal: Proprietary nanocoating protects internal components. == page == ### 4. Performance Benchmarks (Scale 0‑100) AI Processing: The dPhone D1 scores a 98, leveraging the Helios G1 Neural Chip, compared to the competitor's 85. - Battery Life: Optimized power management gives the D1 a score of 95, outlasting the 88 of the rival. Thermal Efficiency: With its advanced liquid‑cooling and titanium chassis, the D1 maintains 92% efficiency under load. Signal Stability: The integrated 6G‑ready modem provides a 94 score for connection reliability in weak signal areas. > dPhone D1 Performance Benchmarks > Performance Score (0‑100) > AI Processing > Battery Life > Thermal Efficiency > Signal Stability **Figure 2.** Performance comparison between the iPhone DI and the current industry leading flagship. The dPhone D1 outperforms its competitor across key performance metrics, excelling in AI processing with a 98/100 score thanks to the Helios G1 Neural Chip. Its optimized battery and thermal management deliver long‑lasting use and stable operation under heavy load. Additionally, the 6G‑ready modem ensures reliable connectivity even in weak signal areas. - End of Document -"""

# Menguji pemotongan teks pada contoh dokumen produk di atas
chunks = chunk_text(doc)

# Menampilkan jumlah potongan teks yang dihasilkan dari dokumen contoh
print(f'Number of chunks: {len(chunks)}')

# Menampilkan seluruh potongan teks hasil pemecahan dokumen
chunks

### Langkah 6: Pemrosesan Chunking Terdistribusi Menggunakan PySpark Pandas UDF
Memanfaatkan Apache Arrow melalui `@pandas_udf` untuk pemrosesan batch terdistribusi berkinerja tinggi pada cluster Spark, kemudian menggunakan `explode()` agar setiap potongan teks menjadi satu baris terpisah dalam DataFrame.


In [0]:
from typing import Iterator
from pyspark.sql.functions import pandas_udf, explode
import pandas as pd

# Mendefinisikan Pandas UDF berbasis batch iterator untuk memproses pemotongan teks secara paralel di cluster Spark
@pandas_udf("array<string>")
def get_chunks(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    for s in batch_iter:
        yield s.apply(chunk_text)

# Menerapkan UDF ke kolom plain_text dan mengekspansi (explode) hasilnya menjadi baris individual per chunk
df_chunks = (silver_df.withColumn("chunk", explode(get_chunks("plain_text")))
                                        .select("path", "chunk")
            )

# Menampilkan DataFrame hasil chunking (path file dan isi potongan teks)
display(df_chunks)

### Langkah 7: Menambahkan ID Unik & Menyimpan ke Tabel Lapisan Gold
Menambahkan kolom `id` unik berbasis 64-bit integer menggunakan `monotonically_increasing_id()` yang berfungsi sebagai *Primary Key* untuk kebutuhan indexing di Vector Search, lalu menyimpan DataFrame ke tabel Delta terkelola `gold_docs_chunked`.


In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Menambahkan kolom "id" unik (64-bit integer) sebagai primary key untuk setiap baris chunk
df_chunks = df_chunks.withColumn("id", monotonically_increasing_id())

# Menyimpan DataFrame ke tabel Delta Lake "gold_docs_chunked" di Unity Catalog
# Tabel ini siap dihubungkan langsung ke Databricks Vector Search Index pada notebook 1.2
df_chunks.write.mode("overwrite").saveAsTable("gold_docs_chunked")